# Plan Analysis

Analyze **`terraform plan -json`** machine-readable UI output.

Expects newline-delimited JSON with `@module: "terraform.ui"` and structured `type` / `hook` / `change` records (for example `refresh_start`, `planned_change`, `resource_drift`).

Capture example:

```bash
terraform plan -json > plan-ui.json
export TERRAFORM_LOG_PATH=plan-ui.json
```

For **`TF_LOG=json`** trace logs, use `plan/performance-analysis.ipynb` instead. Run `whatisit.ipynb` if unsure.

In [ ]:
import sys
from pathlib import Path

for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / "commonlib" / "config.py").is_file():
        _notebooks_root = _root
        break
    if (_root / "notebooks" / "commonlib" / "config.py").is_file():
        _notebooks_root = _root / "notebooks"
        break
else:
    raise RuntimeError(
        "Could not find notebooks/commonlib/. Start Jupyter from notebooks/ "
        "or open a notebook under export/, plan/, apply/, sdk-plan/, or the notebooks root."
    )

if str(_notebooks_root) not in sys.path:
    sys.path.insert(0, str(_notebooks_root))

from commonlib.notebook_setup import setup

setup()

import pandas as pd
import commonlib.prep_plan_output_data as prep_plan_output_data
import commonlib.gencharts as gencharts
import matplotlib.pyplot as plt
import commonlib.config as cfg


In [ ]:
c = cfg.Config()
print(c.TERRAFORM_LOG_PATH)
normalized_records = prep_plan_output_data.load_normalized_records()
df = pd.json_normalize(normalized_records)

if df.empty:
    raise ValueError(
        "No terraform.ui plan records found. Capture with `terraform plan -json` "
        "or use plan/performance-analysis.ipynb for TF_LOG=json trace logs."
    )

print(sorted(df["type"].drop_duplicates().tolist()))

starts = (
    df[df["type"] == "refresh_start"]
    .copy()
    .sort_values(["resource_id", "timestamp"])
)
starts["refresh_start_timestamp"] = pd.to_datetime(starts["timestamp"], utc=True, errors="coerce")
starts["run"] = starts.groupby("resource_id").cumcount() + 1
starts = starts[
    ["resource_id", "run", "refresh_start_timestamp", "resource", "resource_type", "resource_name"]
]

ends = (
    df[df["type"] == "refresh_complete"]
    .copy()
    .sort_values(["resource_id", "timestamp"])
)
ends["refresh_complete_timestamp"] = pd.to_datetime(ends["timestamp"], utc=True, errors="coerce")
ends["run"] = ends.groupby("resource_id").cumcount() + 1
ends = ends[["resource_id", "run", "refresh_complete_timestamp"]]

df_merged_refresh = starts.merge(ends, on=["resource_id", "run"], how="inner")
df_merged_refresh["time_diff_minutes"] = (
    (df_merged_refresh["refresh_complete_timestamp"] - df_merged_refresh["refresh_start_timestamp"])
    .dt.total_seconds()
    / 60
)

df_resource_drift = df[df["type"] == "resource_drift"]
df_planned_change = df[df["type"] == "planned_change"]

## Type Analysis

In [ ]:
gencharts.generate_plt_by_resource_type(df, "refresh_start", top_n=10)
gencharts.generate_plt_by_resource_type(df, "refresh_complete", top_n=10)
gencharts.generate_plt_by_resource_type(df, "resource_drift", top_n=10)
gencharts.generate_plt_by_resource_type(df, "planned_change", top_n=10)

## Duration Analysis

In [ ]:
if not df_merged_refresh.empty:
    gencharts.generate_duration_by_resource_type(df_merged_refresh, metric="total", top_n=10)
    gencharts.generate_duration_by_resource_type(df_merged_refresh, metric="average", top_n=10)
else:
    print("No matched refresh_start/refresh_complete pairs to chart.")

## Longest Refresh Times

In [ ]:
if df_merged_refresh.empty:
    print("No matched refresh_start/refresh_complete pairs to list.")
else:
    display(
        df_merged_refresh[
            ["resource", "resource_type", "refresh_start_timestamp", "refresh_complete_timestamp", "time_diff_minutes", "run"]
        ]
        .copy()
        .sort_values(by="time_diff_minutes", ascending=False)
        .head(20)
    )